# run

> handing one fire to an agent, keeping what it said, and leaving a note

In [ ]:
#| default_exp run

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

This is the edge of pobblebonk. It owns the reminder and not the work. A runner is therefore
anything callable that takes a prompt and gives back text. `AcpRunner` is the one that ships.
Nothing here imports ramabana.

In ACP the editor asks first, so `AcpRunner` is the *client*. It starts an agent, sends one prompt,
and closes. That is the only shape available. An agent cannot start a turn, so a fire cannot appear
inside the editor session you have open. It runs on its own, and the answer reaches you through the
[inbox](04_inbox.ipynb).

`session` is what makes a daily roundup worth having. Keep the id the agent hands back. Tomorrow's
fire then continues the same conversation, and can say "down from last week" because it still has
last week.

In [ ]:
#| export
import asyncio, os, time
from pathlib import Path
from fastcore.all import AttrDict, L, ifnone, patch, store_attr
from pobblebonk.core import clip, now
from pobblebonk.schedule import FIRE, Schedules

In [ ]:
#| export
DFLT_AGENT = 'ramabana-acp'      # what an unattended fire starts; anything speaking ACP will do
DFLT_TIMEOUT = 900.              # seconds one fire gets
WRITE_KINDS = ('edit', 'delete', 'move', 'execute')   # tool kinds a `read` policy refuses

## The prompt a fire sends

The standing instruction, then what the list holds. Nothing else: the agent has its own tools and
its own project, and a prompt full of scaffolding just crowds them out.

In [ ]:
#| export
def prompt_for(sched,          # the schedule row
               items=(),       # its open list items, where it has a list
               fire_at=None,   # which fire this is
               is_late=False   # whether the machine was off when it was due
) -> str:
    'The one prompt a fire sends.'
    out = [str(sched['prompt'])]
    if fire_at:
        when = time.strftime('%Y-%m-%d %H:%M', time.localtime(fire_at))
        out.append(f"\nThis is the {when} run of \"{sched['name']}\"."
                   + (' It is running late, because nothing was awake when it was due.' if is_late else ''))
    if len(items):
        out.append(f"\nThe {sched['needs']} list holds {len(items)} item(s):")
        out += [f"- {i['text']}" for i in items]
    return '\n'.join(out)

In [ ]:
s = dict(name='cart', prompt='Fill my cart.', needs='shopping')
test_eq(prompt_for(s), 'Fill my cart.')
got = prompt_for(s, items=[dict(text='oat milk'), dict(text='yoghurt')], fire_at=0)
test(got, 'shopping list holds 2 item(s)', lambda a,b: b in a)
test(got, '- oat milk', lambda a,b: b in a)

## Runners

A runner takes `(prompt, session, cwd)` and gives back something with `text` and `session`.
`FakeRunner` is that contract in eight lines, and it is what the tests and `pob tick --dry` use.

In [ ]:
#| export
class FakeRunner:
    "A runner that records what it was asked and answers from a script."
    def __init__(self,
                 replies=None,           # answers in order; an `Exception` in the list is raised
                 session:str='fake-1'):  # the id it hands back when it was not given one
        self.replies, self.session, self.asked = L(replies or []), session, []

    def __call__(self, prompt:str, session:str=None, cwd:str=None) -> AttrDict:
        self.asked.append(AttrDict(prompt=prompt, session=session, cwd=cwd))
        out = self.replies.pop(0) if self.replies else 'ok'
        if isinstance(out, Exception): raise out
        return AttrDict(text=out, session=ifnone(session, self.session), refused=L())

In [ ]:
#| export
class AcpRunner:
    """One fire against an ACP agent, started for it and closed after it.

    `policy` is answered here rather than asked of anybody: an unattended fire has no editor to
    put a permission dialog in front of. `read` selects the agent's own reject option and records
    what it refused, so the note says what the fire could not do.
    """
    def __init__(self,
                 cmd:str=DFLT_AGENT,           # the agent to start
                 args=(),                      # its arguments
                 cwd:str=None,                 # the project it opens; this process's directory when unset
                 env:dict=None,                # its environment; this process's when unset
                 policy:str='read',            # `read` refuses writes, `write` allows them
                 timeout:float=DFLT_TIMEOUT):  # seconds one fire gets before it is cut off
        store_attr()

    def __call__(self, prompt:str, session:str=None, cwd:str=None) -> AttrDict:
        'Run one prompt. Blocks, because the queue handler that calls it is on a worker thread.'
        return asyncio.run(asyncio.wait_for(self._go(prompt, session, cwd), self.timeout))

    async def _go(self, prompt, session, cwd):
        import acp
        cl = _Fire(self.policy)
        root = str(ifnone(cwd, self.cwd) or Path.cwd())
        async with acp.spawn_agent_process(cl, self.cmd, *self.args,
                                           env=dict(self.env or os.environ)) as (conn, _):
            await conn.initialize(protocol_version=acp.PROTOCOL_VERSION,
                                  client_capabilities=_Fire.caps)
            sid = await self._session(conn, root, session)
            res = await conn.prompt(session_id=sid, prompt=[acp.text_block(prompt)])
        return AttrDict(text=''.join(cl.text), session=sid, refused=L(cl.refused),
                        stop=getattr(res, 'stop_reason', ''))

    async def _session(self, conn, root, session):
        'Continue the conversation this schedule already had, or start one.'
        if session:
            # an agent that cannot find the saved session is not a reason to skip the fire
            try:
                await conn.load_session(cwd=root, session_id=session)
                return session
            except Exception: pass
        return (await conn.new_session(cwd=root)).session_id

The client side of one fire collects the reply and answers the permission requests. It is built
lazily, so importing this module does not need the `acp` extra installed.

In [ ]:
#| export
class _Fire:
    "The client side of one fire: collect the reply, answer permission from policy."
    caps = None

    def __init__(self, policy:str='read'):
        self.policy, self.text, self.refused = policy, [], []

    async def session_update(self, session_id, update, **kw):
        from acp.schema import AgentMessageChunk
        if isinstance(update, AgentMessageChunk):
            if (t := getattr(getattr(update, 'content', None), 'text', None)): self.text.append(t)

    async def request_permission(self, session_id, tool_call, options, **kw):
        from acp.schema import AllowedOutcome, DeniedOutcome, RequestPermissionResponse
        kind = str(getattr(tool_call, 'kind', '') or '')
        ok = self.policy == 'write' or kind not in WRITE_KINDS
        if not ok:
            self.refused.append(f"{getattr(tool_call, 'title', '') or kind} ({kind})")
        want = 'allow' if ok else 'reject'
        opt = next((o for o in (options or []) if str(o.option_id).startswith(want)), None)
        if opt is None:
            if ok: raise RuntimeError('the agent offered no option this fire may take')
            return RequestPermissionResponse(outcome=DeniedOutcome(outcome='cancelled'))
        return RequestPermissionResponse(outcome=AllowedOutcome(outcome='selected',
                                                                option_id=opt.option_id))

    async def write_text_file(self, session_id, path, content, **kw):
        import acp
        raise acp.RequestError.method_not_found('fs/write_text_file')

    async def read_text_file(self, session_id, path, line=None, limit=None, **kw):
        import acp
        raise acp.RequestError.method_not_found('fs/read_text_file')

## The handler

What the queue runs when a fire comes up. It reads the schedule, gathers the list, runs it, keeps
the session, marks the items used and leaves the note. A fire that fails raises, so the queue
retries it and the items stay on the list.

In [ ]:
#| export
def fire_handler(runner,        # `(prompt, session, cwd) -> AttrDict(text=, session=)`
                 scheds:Schedules,
                 inbox,         # where the answer is left
                 lists=None,    # the lists a schedule with `needs` reads
                 cwd:str=None   # the project a fire opens
):
    'The queue handler for one schedule fire.'
    def run_fire(payload:dict) -> dict:
        s = scheds.get(payload['schedule'])
        if s is None: return dict(skipped=True, why='the schedule was dropped')
        items = lists.items(s['needs']) if (s['needs'] and lists is not None) else L()
        if s['needs'] and not items: return dict(skipped=True, why=f"the {s['needs']} list is empty")
        out = AttrDict(runner(prompt_for(s, items, payload.get('fire_at'), payload.get('late')),
                             session=s['session'], cwd=cwd))
        text, sid = clip(out.get('text') or ''), out.get('session')
        if sid and sid != s['session']: scheds.remember(s['id'], sid)
        if len(items): lists.used(items.attrgot('id'), by=s['name'])
        refused = L(out.get('refused') or [])
        body = text + (f"\n\nRefused under the `{s['policy']}` policy: " + ', '.join(refused)
                       if refused else '')
        inbox.note('fire', s['name'], body, ref=s['id'],
                   meta=dict(late=bool(payload.get('late')), items=len(items), refused=list(refused)))
        return dict(text=text, items=len(items), session=sid, refused=list(refused))
    return run_fire

def attach(q,                 # the queue whose fires this handles
           runner,            # what runs one fire
           scheds:Schedules,
           inbox,
           lists=None,
           cwd:str=None):
    'Register the fire handler on `q` and give it back.'
    q.register(FIRE, fire_handler(runner, scheds, inbox, lists, cwd))
    return q

Wednesday night, end to end, with a runner that does not need a model:

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pobblebonk.core import vault
from pobblebonk.inbox import Inbox
from pobblebonk.lists import Lists
from pobblebonk.queue import Queue

kol = ZoneInfo('Asia/Kolkata')
wed = datetime(2026, 8, 26, 20, 1, tzinfo=kol).timestamp()

db = vault()
sch, q, ib, ls = Schedules(db), Queue(db), Inbox(db), Lists(db)
run = FakeRunner(['added 6 items, no oat milk'], session='cart-session')
attach(q, run, sch, ib, ls)

ls.add('shopping', 'yoghurt, the greek one')
ls.add('shopping', 'oat milk')
sch.add('cart', 'Fill my cart from the shopping list.', cron='0 20 * * 3', tz='Asia/Kolkata',
        needs='shopping', policy='write', start=wed - 60)

test_eq(len(sch.tick(q, at=wed, inbox=ib, have=ls.count).fired), 1)
ran = q.drain('tick', at=wed)
test_eq(ran[0].status, 'ok')
test_eq(ran[0].result['items'], 2)

The prompt carried the list, the note carries the answer, the items are used, and the session is
kept for next Wednesday:

In [ ]:
test(run.asked[0].prompt, '- oat milk', lambda a,b: b in a)
test_eq(ib.drain('leela').attrgot('title'), ['cart'])
test_eq(ib.notes()[0]['body'], 'added 6 items, no oat milk')
test_eq(ls.count('shopping'), 0)
test_eq(sch.get('cart')['session'], 'cart-session')

Next week continues the same conversation rather than starting a new one:

In [ ]:
ls.add('shopping', 'bread')
sch.tick(q, at=wed + 7*86400, inbox=ib, have=ls.count)
q.drain('tick', at=wed + 7*86400)
test_eq(run.asked[1].session, 'cart-session')

A fire that raises is retried and the list is untouched, which is why nothing claims the milk up
front:

In [ ]:
db2 = vault()
s2, q2, ib2, ls2 = Schedules(db2), Queue(db2), Inbox(db2), Lists(db2)
attach(q2, FakeRunner([RuntimeError('the shop was down')]), s2, ib2, ls2)
ls2.add('shopping', 'milk')
s2.add('cart', 'Fill my cart.', every='1h', needs='shopping', start=0)
s2.tick(q2, at=1, inbox=ib2, have=ls2.count)
test_eq(q2.drain('tick', at=1)[0].status, 'error')
test_eq(ls2.count('shopping'), 1)     # still there for the retry
test_eq(ib2.count('fire'), 0)         # and nothing was reported as an answer

A schedule whose list emptied between the tick and the run is skipped rather than failed, because
there is nothing wrong and nothing to say:

In [ ]:
ls2.used(ls2.items('shopping').attrgot('id'))
q2.retry(q2.jobs(kind=FIRE)[0]['id'], at=2)
got = q2.drain('tick', at=2)
test_eq(got[0].status, 'skipped')
test_eq(got[0].result['why'], 'the shopping list is empty')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()